In [29]:
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
from scipy.optimize import linear_sum_assignment
from typing import List, Tuple, Dict, Optional, Iterable, Union

from collections import Counter, defaultdict
from itertools import combinations

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from collections import defaultdict
import math
import re

In [2]:
save_path = "/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/text-embedding-3-small/embeddings.npz"

In [3]:
# 불러오기
data = np.load(save_path, allow_pickle=True)
X, y_true, texts = data["X"], data["labels"], data["texts"]
print("Loaded:", X.shape, y_true.shape, texts.shape)

Loaded: (18235, 1536) (18235,) (18235,)


### soft k-means

In [4]:
def soft_kmeans(X, n_clusters, beta, max_iter=100, tol=1e-4, random_state=42):
    np.random.seed(random_state)
    N, D = X.shape
    
    # 1. 초기 중심 랜덤 선택
    indices = np.random.choice(N, n_clusters, replace=False)
    centers = X[indices]

    for it in range(max_iter):
        # 2. E-step: soft assignment 확률 계산
        dists = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2) ** 2  # (N, K)
        exp_dists = np.exp(-beta * dists)
        probs = exp_dists / exp_dists.sum(axis=1, keepdims=True)  # (N, K)

        # 3. M-step: 중심 업데이트 (가중 평균)
        new_centers = (probs.T @ X) / probs.sum(axis=0)[:, None]

        # 4. 수렴 체크
        if np.linalg.norm(new_centers - centers) < tol:
            break
        centers = new_centers

    return probs, centers

In [5]:
P_soft, C_soft = soft_kmeans(X, n_clusters=20, beta=30.0) 

In [6]:
soft_pred = np.argmax(P_soft, axis=1)

In [7]:
len(soft_pred)

18235

### hungarian match

In [39]:
def hungarian_match_from_soft(
    y_true: List,           # 정답 라벨(숫자/문자열 모두 가능)
    P_soft: np.ndarray,     # (N, K) 소프트 멤버십
) -> Tuple[Dict[int, int], np.ndarray, np.ndarray]:
    """
    P_soft만으로 소프트 컨퓨전 행렬을 만들어 Hungarian 매칭 수행.
    반환:
      mapping: dict {pred_cluster_index -> true_label_original}
      y_pred_hard: (N,) argmax로 얻은 하드 예측 라벨(정렬 전, pred 기준)
      y_pred_aligned: (N,) mapping을 적용하여 true 라벨 체계로 정렬된 라벨
    """
    y_true = np.asarray(y_true)
    N, K = P_soft.shape

    # 1) true 라벨을 0..T-1로 압축
    true_ids, y_true_comp = np.unique(y_true, return_inverse=True)
    T = true_ids.size

    # 2) 소프트 컨퓨전 행렬 S: (T, K)
    # S[t, k] = sum_i P_soft[i,k] where true class of i = t
    S = np.zeros((T, K), dtype=np.float64)
    for t in range(T):
        idx = np.where(y_true_comp == t)[0]
        if idx.size > 0:
            S[t] = P_soft[idx].sum(axis=0)

    # 3) Hungarian (최대화 → 비용 = -S 최소화)
    cost = -S
    row_ind, col_ind = linear_sum_assignment(cost)  # row: true(t), col: pred(k)

    # 4) pred(k) -> true(원본라벨) 매핑 딕셔너리 구성
    mapping = {}
    for t, k in zip(row_ind, col_ind):
        mapping[int(k)] = true_ids[int(t)]  # pred k → true label(original)

    # 5) 참고용: 하드 예측, 정렬된 라벨
    y_pred_hard = P_soft.argmax(axis=1)                 # (N,)
    y_pred_aligned = np.array([mapping.get(int(k), k) for k in y_pred_hard])

    return mapping, y_pred_hard, y_pred_aligned


def reorder_by_mapping(
    P_soft: np.ndarray,                 # (N, K)
    C_soft: Optional[np.ndarray],       # (K, D) 또는 None
    mapping: Dict[int, int],            # pred k → true label(original)
) -> Tuple[np.ndarray, Optional[np.ndarray], Dict[int, int]]:
    """
    mapping을 이용해 P_soft의 열, C_soft의 행을 'true 라벨 오름차순(0..T-1)' 기준으로 정렬.
    true 라벨이 숫자/문자열 혼용일 수 있어 우선 true 라벨을 정렬 가능한 순서로 매핑합니다.
    반환:
      P_aligned, C_aligned, k_old_to_new  (pred k → 정렬 후 열 인덱스)
    """
    K = P_soft.shape[1]
    # true 라벨 원본값들의 정렬 순서 결정
    true_labels = list(mapping.values())
    # 숫자/문자열 섞임 가능성을 대비해 문자열로 정렬 키를 생성
    order = np.argsort(np.array([str(v) for v in true_labels], dtype=object))

    # 정렬 후의 열 인덱스(new) -> 정렬 전의 pred k(old)
    k_old_sorted = np.array(list(mapping.keys()))[order]
    # 정렬 후 열 이름이 0..K-1 이 되도록 old→new 역매핑도 보관
    k_old_to_new = {int(old_k): int(new_k) for new_k, old_k in enumerate(k_old_sorted)}

    # P 재정렬 (열)
    P_aligned = P_soft[:, k_old_sorted]

    # C 재정렬 (행)
    C_aligned = None
    if C_soft is not None:
        C_aligned = C_soft[k_old_sorted, :]

    return P_aligned, C_aligned, k_old_to_new

In [43]:
mapping, y_pred_hard, y_pred_aligned = hungarian_match_from_soft(y_true, P_soft) # mapping : 매칭결과 (전:후), y_pred_hard : 클러스터결과, y_pred_aligned : 헝가리안매치 결과

In [49]:
P_aligned, C_aligned, _ = reorder_by_mapping(P_soft, C_soft, mapping)

In [48]:
y_pred_aligned

array([10,  3, 17, ...,  3, 14,  7])

### 클러스터 결과 저장

In [50]:
df = pd.read_csv('/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/preprocessed_20ng.csv')

In [52]:
df["cluster_label"] = y_pred_aligned

In [57]:
df.to_csv('/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/cluster_20ng.csv', index=False)

### 소프트 맴버십 저장

In [58]:
def save_soft_results(
    path: str,
    P: np.ndarray,
    C: np.ndarray,
    mapping: dict,
    y_pred_hard: np.ndarray,
    y_pred_aligned: np.ndarray
):
    np.savez_compressed(
        path,
        P=P,
        C=C,
        y_pred_hard=y_pred_hard,
        y_pred_aligned=y_pred_aligned,
        mapping=np.array(list(mapping.items()), dtype=np.int32)  # dict은 npz에 직접 안 들어가서 튜플 배열로
    )
    print(f"[saved] {path} | P={P.shape}, C={C.shape}")

In [59]:
save_soft_results("/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/text-embedding-3-small/soft_results.npz", P_aligned, C_aligned, mapping, y_pred_hard, y_pred_aligned)

[saved] /home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/text-embedding-3-small/soft_results.npz | P=(18235, 20), C=(20, 1536)


### top words

In [12]:
def _default_tokenize(text: str, min_word_len: int = 2) -> List[str]:
    # 한글 포함하려면 패턴을 r"[가-힣a-zA-Z]+" 로 교체
    tokens = re.findall(r"[a-zA-Z]+", text.lower())
    return [t for t in tokens if len(t) >= min_word_len]

def _generate_ngrams(tokens: List[str], ngram_range: Tuple[int, int]) -> Iterable[str]:
    nmin, nmax = ngram_range
    for n in range(nmin, nmax + 1):
        if n == 1:
            for tok in tokens:
                yield tok
        else:
            for i in range(len(tokens) - n + 1):
                yield "_".join(tokens[i:i+n])

def build_topics_from_clusters_ctfidf(
    texts: Union[List[str], List[List[str]]],  # 문자열 문서 or 토큰 리스트
    labels_pred: List[int],
    topn: int = 10,
    min_word_len: int = 4,
    ngram_range: Tuple[int, int] = (1, 1),
    include_noise: bool = False,
    tokenizer: Optional[callable] = None,      # 문자열 문서일 때만 사용
    stop_words: Optional[Iterable[str]] = None,
    return_scores: bool = False,
    apply_length_filter_on_pretokenized: bool = False,  # 토큰이 이미 주어져도 길이 필터 적용할지
    lowercase_pretokenized: bool = False,               # 사전 토큰에도 소문자화 적용할지
    hangul_mode: bool = False                           # True면 기본 토큰화 정규식을 한글포함으로
) -> List[List[str]]:
    """
    c-TF-IDF로 각 클러스터(토픽)의 상위 단어를 추출.
    texts:
      - List[str]: 미토큰화 문서들 → tokenizer 또는 기본 토큰화 수행
      - List[List[str]]: 이미 토큰화된 문서들 → 그대로 사용(옵션 필터만 적용)
    """
    assert len(texts) == len(labels_pred), "texts와 labels_pred 길이가 달라요."

    # 0) stopwords
    stop_set = set(stop_words) if stop_words is not None else None

    # 1) 클러스터별 문서 수집
    cluster_docs: Dict[int, List[int]] = defaultdict(list)  # 문서 인덱스만 저장
    for idx, lab in enumerate(labels_pred):
        if lab == -1 and not include_noise:
            continue
        cluster_docs[lab].append(idx)

    if not cluster_docs:
        return []

    # 2) 토큰 소스 준비 (문자열인지/토큰인지 자동 판별)
    #    문자열 문서 → 토큰화, 토큰 문서 → 그대로
    is_pretokenized = isinstance(texts[0], (list, tuple))
    if not is_pretokenized:
        # 기본 토크나이저 준비
        if tokenizer is None:
            if hangul_mode:
                def tokenizer(x: str) -> List[str]:
                    toks = re.findall(r"[가-힣a-zA-Z]+", (x or "").lower())
                    return [t for t in toks if len(t) >= min_word_len]
            else:
                def tokenizer(x: str) -> List[str]:
                    return _default_tokenize(x or "", min_word_len=min_word_len)

    # 3) 클러스터별 토큰 나열
    cluster_tokens: Dict[int, List[str]] = {}
    for cid, doc_indices in cluster_docs.items():
        toks: List[str] = []
        for di in doc_indices:
            if is_pretokenized:
                # 이미 토큰화된 경우
                ts = list(texts[di])  # shallow copy
                if lowercase_pretokenized:
                    ts = [t.lower() for t in ts]
                if apply_length_filter_on_pretokenized and min_word_len > 1:
                    ts = [t for t in ts if len(t) >= min_word_len]
            else:
                # 문자열 문서 → 토큰화
                ts = tokenizer(texts[di])

            # stopwords 제거
            if stop_set:
                ts = [w for w in ts if w not in stop_set]

            # n-gram 전개
            for tok in _generate_ngrams(ts, ngram_range):
                if (not stop_set) or (tok not in stop_set):
                    toks.append(tok)

        cluster_tokens[cid] = toks

    # 4) 어휘 사전
    vocab: Dict[str, int] = {}
    for toks in cluster_tokens.values():
        for w in toks:
            if w not in vocab:
                vocab[w] = len(vocab)

    V = len(vocab)
    C = len(cluster_tokens)
    if V == 0:
        # 클러스터 수만큼 빈 리스트 반환
        return [[] for _ in range(C)]

    # 5) TF (클러스터-단어 카운트)
    counts = {cid: [0] * V for cid in cluster_tokens.keys()}
    total_words_per_class = {}
    for cid, toks in cluster_tokens.items():
        row = counts[cid]
        for w in toks:
            row[vocab[w]] += 1
        total_words_per_class[cid] = sum(row)

    # 6) c-TF-IDF 계수
    # tf_t: 전체 클러스터에서의 term 총 빈도
    tf_t = [0] * V
    for row in counts.values():
        for j in range(V):
            tf_t[j] += row[j]

    A = (sum(total_words_per_class.values()) / float(C)) if C > 0 else 0.0
    inv_class_part = [0.0] * V  # log(1 + A / tf_t)
    for j in range(V):
        inv_class_part[j] = math.log(1.0 + (A / tf_t[j])) if tf_t[j] > 0 else 0.0

    inv_vocab = {idx: w for w, idx in vocab.items()}

    # 7) 정렬된 클러스터 순회 & 상위 단어 반환
    results: List[List[str]] = []
    for cid in sorted(cluster_tokens.keys()):
        row = counts[cid]
        scores = [(j, row[j] * inv_class_part[j]) for j in range(V) if row[j] > 0]
        scores.sort(key=lambda x: x[1], reverse=True)
        top_items = scores[:topn]
        if return_scores:
            results.append([(inv_vocab[j], float(s)) for j, s in top_items])
        else:
            results.append([inv_vocab[j] for j, _ in top_items])

    return results


### 성능 비교

In [13]:
# -------------------------
# External metrics
# -------------------------
def purity_p1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    P@1 (Purity): 각 예측 클러스터에서 최빈 진짜 라벨 개수를 합산 / 전체 샘플 수
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    assert y_true.shape[0] == y_pred.shape[0]
    N = y_true.shape[0]
    total = 0
    for k in np.unique(y_pred):
        mask = (y_pred == k)
        if mask.sum() == 0:
            continue
        majority = Counter(y_true[mask]).most_common(1)[0][1]
        total += majority
    return total / N

def ext_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "P1": purity_p1(y_true, y_pred),
        "ARI": adjusted_rand_score(y_true, y_pred),
        "NMI": normalized_mutual_info_score(y_true, y_pred)
    }
# -------------------------
# Internal metrics
# -------------------------
def topic_diversity(topics_words: List[List[str]], topn: int = 10) -> Optional[float]:
    """
    TD: 고유 단어 수 / (K * topn)
    topics_words: 각 토픽의 상위 단어 리스트
    """
    if not topics_words:
        return None
    trimmed = [tw[:topn] for tw in topics_words if tw]
    if not trimmed:
        return None
    K = len(trimmed)
    all_words = [w for tw in trimmed for w in tw]
    if K * topn == 0:
        return None
    return len(set(all_words)) / (K * topn)

# -------------------------
# Wrapper: evaluate multiple models
# -------------------------
def evaluate_models(
    y_true: np.ndarray,
    preds: Dict[str, np.ndarray],
    topics_words: Optional[Dict[str, List[List[str]]]] = None,
    docs_tokens: Optional[List[List[str]]] = None,
    topn: int = 10
) -> Dict[str, Dict[str, Optional[float]]]:
    """
    preds: {"k": k_label, "p": p_label, "s": s_label, "f": f_label, ...}
    topics_words: {"k": [[w,...],[...],...], "p": ...}  # 모델별 토픽 상위단어 리스트
    docs_tokens: 토큰화된 전체 문서(UMass 계산용)
    """
    results = {}
    for name, y_pred in preds.items():
        # External
        e = ext_metrics(y_true, y_pred)

        # Internal (모델별 토픽 단어가 제공된 경우만 계산)
        td = None
        tc_umass = None
        if topics_words and name in topics_words and topics_words[name]:
            td = topic_diversity(topics_words[name], topn=topn)

        results[name] = {
            "P1": e["P1"],
            "ARI": e["ARI"],
            "NMI": e["NMI"],
            "TD": td
        }
    return results

In [14]:
# 모델별 토픽 단어 추출
topics_words = {
    "s": build_topics_from_clusters_ctfidf(texts, s_label, topn=10),
}

# 평가 실행
metrics = evaluate_models(
    y_true,
    preds={
        "s": s_label
    },
    topics_words=topics_words,  # 모델별 토픽 단어 딕셔너리 전달
    docs_tokens=None,           # 필요하면 토큰화된 문서 리스트 넣기
    topn=10
)

# 결과 출력
for model, scores in metrics.items():
    print(model, scores)


s {'P1': 0.5946257197696737, 'ARI': 0.4217166957613969, 'NMI': 0.5681847387504138, 'TD': 0.88}


### 토픽 단어 저장

In [15]:
with open("/home/ys0660/2507Sub/textclustering/0915/Topicker/data/20NG/text-embedding-3-small/topic_words.txt", "w", encoding="utf-8") as f:
    for topic_id, words in enumerate(topics_words["s"]):
        f.write(f"Topic {topic_id}: {', '.join(words)}\n")